# Beneficial ESS profile — 100-run experiment — **B1_premium_fully_governed**

**Profile** (supplier, manufacturer, inspector, distributor, regulator) = `['high', 'fullQC', 'strict', 'accept', 'auditH']`.

Premium fully-governed regime: top inputs, full QC, strict inspection, lots accepted, strong audit. The gold-standard high-assurance supply chain (pharma, aerospace, medical devices). Strongest stability margin of all ESS.

This profile was identified as a genuine ESS in the 576-profile scan (all eigenvalues negative, all validation replicator simulations converging). Here we run the **full 100-run** experiment for this single profile under paper-scale settings, and report **strict success** (a run passes only if all validation replicator starts converge). Run top-to-bottom on a GPU.

# Generic $N$-Player Hierarchical MARL Framework — **SAC branch**

This notebook is the **SAC-only** half of the original combined notebook, split so the
SAC outer-loop search and the Bayesian-optimization (BO) baseline can be run
**simultaneously in two separate notebooks/kernels** (e.g. two Colab runtimes, or two
local Jupyter kernels), instead of sequentially in one notebook.

Contains: game definition, payoff tensor, agent networks, inner-loop PPO, outer-loop
reward, the SAC meta-optimizer, replicator + numerical-Jacobian validation, the SAC run,
SAC validation, the inner-loop trajectory/cycling check (Section 15, built on the SAC
results), and the wall-clock scaling benchmark (Section 16, optimizer-independent).

Run top to bottom independently of the BO notebook. Both write CSVs to
`./results_5player/`; see the note at the end of this notebook for combining results
into the SAC-vs-BO comparison table afterward.

# Generic $N$-Player Hierarchical MARL Framework for High-Dimensional Evolutionary Games
### Larger-scale validation for the IJPR revision (Reviewer 1.1 / 1.2 / 1.3 and Reviewer 3 major #1)

This notebook generalizes the three-player framework to **any number of players, each with any number of strategies**, and applies it to a **five-player, multi-strategy supply-chain quality-governance game** whose joint strategy space ($3\times4\times3\times4\times4 = 576$ pure profiles, a $13\times 13$ Jacobian) is far beyond hand analysis.

**What is preserved from the original code (so results stay comparable and fast):**
- the independent-PPO **inner loop** implemented with `jax.lax.scan` over episodes,
- **`jax.vmap`** over the agent axis (this is what replaces the three hard-coded reward functions with one generic vectorized update),
- **`jax.jit`** compilation of the whole inner loop,
- **Gumbel-argmax** action sampling,
- the SAC-style **outer loop** that proposes game parameters,
- validation by **forward replicator simulation** and a **numerical Jacobian** (the two checks that survive when no closed-form Jacobian exists).

**What is new:** the game is defined by a data-driven **payoff tensor** of shape `(n_players, k_1, ..., k_N)` built from economically meaningful parameters, and a per-agent **valid-strategy mask** so players can have different strategy counts. Nothing about the acceleration structure changes.

> **Note on early stopping.** Following Reviewer 3 (major #4), early stopping is now an **explicit boolean flag** (`use_early_stopping`). In all reported runs it is **disabled** (we run the full epoch budget), which is why the original patience was set to a large number. Both behaviours are now standard and documented.


## 1. Imports and global configuration
Standard JAX / Optax / NumPy stack. We set 64-bit precision off (32-bit is faster on GPU and sufficient here) and fix a master seed for reproducibility.

In [1]:
import jax, jax.numpy as jnp
from jax import lax
from functools import partial
import numpy as np
import pandas as pd
import itertools, time, os
from scipy.integrate import solve_ivp

MASTER_SEED = 42
np.random.seed(MASTER_SEED)
print("JAX", jax.__version__, "| devices:", jax.devices())


JAX 0.10.1 | devices: [CpuDevice(id=0)]


### 1b. Run configuration and output folders

One switch, `PAPER_SCALE`, toggles between a fast CPU demo and the full paper-scale run. **Every experiment writes its results to CSV** under `./results_5player/`, so you can re-run any single block, hand the CSVs back, and the figures/tables can be regenerated without re-running the search. Set `PAPER_SCALE = True` on a GPU machine (Colab is fine) to produce the numbers reported in the paper.

The paper-scale values are taken **directly from the original E3/E4/E7/E8 notebooks' executed calls**, so the larger game is run under the same protocol as the small game: `n_runs = 100`, outer epochs `= 200`, inner PPO episodes `= 3000`, `patience = 1000` (early stopping effectively disabled, i.e. the full outer budget always runs), validation seeds `= 5`, warmup `= 5`.

- Demo (`PAPER_SCALE=False`): a handful of runs, short budgets, finishes in minutes on CPU.
- Paper (`PAPER_SCALE=True`): the settings above; run on a GPU.

In [2]:
PAPER_SCALE = True   # <--- set True on GPU for the reported numbers

# Paper-scale settings match the actual calls in the original E3/E4/E7/E8 notebooks:
#   n_runs=100, n_epochs(outer)=200, ppo_episodes(inner)=3000,
#   patience=1000 (early stopping effectively disabled), n_val_seeds=5, warmup_epochs=5.
if PAPER_SCALE:
    CFG = dict(N_RUNS=100, N_OUTER=500, N_EPISODES=3000,
               N_VAL_STARTS=20, PATIENCE=1000, WARMUP=5)
else:
    CFG = dict(N_RUNS=8,   N_OUTER=12,  N_EPISODES=800,
               N_VAL_STARTS=12, PATIENCE=1000, WARMUP=5)

OUTDIR = "results_5player"
os.makedirs(OUTDIR, exist_ok=True)
def save_csv(df, name):
    path = os.path.join(OUTDIR, name)
    df.to_csv(path, index=False)
    print(f"  saved -> {path}  ({len(df)} rows)")
    return path
print("PAPER_SCALE =", PAPER_SCALE, "| config:", CFG)


PAPER_SCALE = True | config: {'N_RUNS': 100, 'N_OUTER': 500, 'N_EPISODES': 3000, 'N_VAL_STARTS': 20, 'PATIENCE': 1000, 'WARMUP': 5}


## 2. Game specification: a five-player supply-chain quality-governance game

This is a genuine multi-tier generalization of the three-player e-commerce model, with a **variable number of strategies per player (3 or 4)** as required. The tiers are:

| # | Player | Strategies ($k$) | Meaning |
|---|--------|------------------|---------|
| 1 | **Supplier** | 3: high / standard / substandard | input grade |
| 2 | **Manufacturer** | 4: full-QC / partial-QC / minimal-QC / no-QC | internal quality-control effort |
| 3 | **Inspector** (buyer-side QC) | 3: strict / sampling / slack | incoming inspection intensity |
| 4 | **Distributor / retailer** | 4: accept / re-inspect / return / recall | lot decision on the received batch |
| 5 | **Regulator / certifier** | 4: high / medium / low / no audit | external oversight and penalties |

The joint pure-strategy space is $3\times4\times3\times4\times4 = 576$ profiles; the reduced replicator system is **13-dimensional**, and its Jacobian at each of the $576$ corners is a $13\times13$ symbolic matrix. By Abel–Ruffini the eigenvalue conditions have no general closed form, so this instance is **analytically intractable by hand**, which is exactly the regime the framework targets. (The original published model has 3 players, 2 strategies each: 8 profiles and a $3\times3$ Jacobian.)

**Economic logic (payoffs are not random).** A unit's genuine quality $q$ is the product of the supplier-grade quality and the manufacturer-effort quality. A defect ($1-q$) is caught with probability determined jointly by the inspector, distributor, and regulator (a complementary detection cascade). A **caught** defect triggers penalties on the upstream players; an **escaped** defect triggers reputation loss $W$, buyer disutility $B$, and downstream losses. Each player pays its own action cost. This produces the same structural tension as the small game (cutting quality saves cost but risks penalty and reputation), now across five interacting tiers with finer-grained strategy choices.

In [3]:
# ---- strategy labels (VARIABLE count per player: 3 or 4) ----
STRAT = {
 'supplier':    ['high','standard','sub'],                     # 3
 'manufacturer':['fullQC','partQC','minQC','noQC'],            # 4
 'inspector':   ['strict','sampling','slack'],                 # 3
 'distributor': ['accept','reinspect','return','recall'],      # 4
 'regulator':   ['auditH','auditM','auditL','noAudit'],        # 4
}
PLAYERS   = list(STRAT.keys())
N_PLAYERS = len(PLAYERS)
STRAT_COUNTS = [len(STRAT[p]) for p in PLAYERS]
K_MAX = max(STRAT_COUNTS)
print("players:", PLAYERS)
print("strategy counts:", STRAT_COUNTS, "| K_max:", K_MAX)
print("joint pure profiles:", int(np.prod(STRAT_COUNTS)))
print("reduced Jacobian dimension:", sum(k-1 for k in STRAT_COUNTS))

# valid-strategy mask: 1 where a strategy exists, 0 where padded to K_MAX
VALID_MASK = np.zeros((N_PLAYERS, K_MAX))
for i,k in enumerate(STRAT_COUNTS):
    VALID_MASK[i,:k] = 1.0
VALID_MASK = jnp.array(VALID_MASK)


players: ['supplier', 'manufacturer', 'inspector', 'distributor', 'regulator']
strategy counts: [3, 4, 3, 4, 4] | K_max: 4
joint pure profiles: 576
reduced Jacobian dimension: 13


## 3. Parameter space $\theta$ and bounds

The outer loop searches over the economic parameters below (the analogue of the 13-dimensional $\theta$ in the small game, now 21-dimensional). Bounds are chosen to keep costs ordered (`Cs_h > Cs_s > Cs_b`, etc.) and economically sensible. As in the original framework, **no eigenvalue information ever enters the search** (that would be data leakage and is unavailable at scale).

In [4]:
PARAM_NAMES = ['P','Cs_h','Cs_s','Cs_b','Cm_f','Cm_p','Cm_m','Cm_n',
               'Ci_s','Ci_p','Ci_0','Ca_h','Ca_m','Ca_l','Cd_r','Cd_ret','Cd_rec',
               'W','B','F_m','F_s','I','R_p','R_c','Vq','G','Sc']
# Wider, economically-ordered bounds. Vq (quality premium), G (governance revenue), and
# Sc (inspection-service revenue) are game parameters that reward realized quality and oversight.
PARAM_BOUNDS = {
 'P':(120,450),'Cs_h':(60,140),'Cs_s':(35,95),'Cs_b':(12,55),
 'Cm_f':(35,110),'Cm_p':(20,75),'Cm_m':(10,50),'Cm_n':(3,28),
 'Ci_s':(20,90),'Ci_p':(8,45),'Ci_0':(1,15),
 'Ca_h':(20,90),'Ca_m':(12,55),'Ca_l':(5,30),
 'Cd_r':(5,40),'Cd_ret':(12,60),'Cd_rec':(25,95),
 'W':(30,220),'B':(20,180),'F_m':(40,320),'F_s':(20,200),
 'I':(5,70),'R_p':(10,70),'R_c':(50,220),
 'Vq':(20,220),'G':(10,180),'Sc':(10,150),
}
N_PARAMS = len(PARAM_NAMES)
LOWER = jnp.array([PARAM_BOUNDS[n][0] for n in PARAM_NAMES])
UPPER = jnp.array([PARAM_BOUNDS[n][1] for n in PARAM_NAMES])
# Logical ordering constraints enforced by monotone repair after scaling:
#   P > Cs_h ; Cs_h>Cs_s>Cs_b ; Cm_f>Cm_p>Cm_m>Cm_n ; Ci_s>Ci_p>Ci_0 ;
#   Ca_h>Ca_m>Ca_l ; Cd_rec>Cd_ret>Cd_r
_ORDER_GROUPS = [['Cs_h','Cs_s','Cs_b'],['Cm_f','Cm_p','Cm_m','Cm_n'],
                 ['Ci_s','Ci_p','Ci_0'],['Ca_h','Ca_m','Ca_l'],
                 ['Cd_rec','Cd_ret','Cd_r']]
print("theta dimension:", N_PARAMS)

theta dimension: 27


## 4. Building the payoff tensor from $\theta$

`build_payoff_tensor(theta)` returns an array of shape `(N_PLAYERS, K_MAX, ..., K_MAX)` giving every player's payoff under every joint pure profile (padded strategies are masked). It is written with NumPy for clarity and then converted to a JAX array; it is called **once per outer step** (cheap: 576 profiles). This is the single function a user rewrites to define a different game; the entire MARL machinery downstream is game-agnostic.

The detection cascade and quality composition make the tensor economically coherent rather than arbitrary.

In [5]:
Q_SUP = {'high':0.97,'standard':0.80,'sub':0.45}
Q_MFR = {'fullQC':0.97,'partQC':0.85,'minQC':0.70,'noQC':0.55}
D_INS = {'strict':0.85,'sampling':0.50,'slack':0.10}
D_DIS = {'accept':0.0,'reinspect':0.55,'return':0.85,'recall':1.0}
D_REG = {'auditH':0.55,'auditM':0.35,'auditL':0.18,'noAudit':0.0}
_A_LVL   = {'auditH':1.0,'auditM':0.6,'auditL':0.3,'noAudit':0.0}
_INS_LVL = {'strict':1.0,'sampling':0.55,'slack':0.15}

def build_payoff_tensor(th):
    c_sup={'high':th['Cs_h'],'standard':th['Cs_s'],'sub':th['Cs_b']}
    c_mfr={'fullQC':th['Cm_f'],'partQC':th['Cm_p'],'minQC':th['Cm_m'],'noQC':th['Cm_n']}
    c_ins={'strict':th['Ci_s'],'sampling':th['Ci_p'],'slack':th['Ci_0']}
    c_reg={'auditH':th['Ca_h'],'auditM':th['Ca_m'],'auditL':th['Ca_l'],'noAudit':0.0}
    c_dis={'accept':0.0,'reinspect':th['Cd_r'],'return':th['Cd_ret'],'recall':th['Cd_rec']}
    P,W,B=th['P'],th['W'],th['B']; Fm,Fs,I=th['F_m'],th['F_s'],th['I']
    Rp,Rc=th['R_p'],th['R_c']; Vq,G,Sc=th['Vq'],th['G'],th['Sc']
    payoff=np.zeros((N_PLAYERS,)+tuple([K_MAX]*N_PLAYERS))
    for idx in itertools.product(*[range(STRAT_COUNTS[i]) for i in range(N_PLAYERS)]):
        s,m,ins,dis,reg=[STRAT[PLAYERS[i]][idx[i]] for i in range(N_PLAYERS)]
        q=Q_SUP[s]*Q_MFR[m]; defect=1-q
        pdet=1-(1-D_INS[ins])*(1-D_DIS[dis])*(1-D_REG[reg])
        escaped=defect*(1-pdet); caught=defect*pdet
        qbonus=Vq*q                                   # market pays for realized quality
        sup = P + 0.5*qbonus - c_sup[s] - caught*Fs - escaped*0.5*W
        man = P + qbonus - c_mfr[m] - caught*Fm - escaped*W
        insp= Rp + Sc*_INS_LVL[ins] - c_ins[ins] + caught*0.3*Fm - escaped*B
        dist= Rc + 0.5*qbonus - c_dis[dis] - escaped*B + (I*defect if dis in ('return','recall') else 0)
        regu= G*_A_LVL[reg] - c_reg[reg] + caught*0.25*(Fm+Fs) - escaped*0.5*B
        payoff[(slice(None),)+idx]=[sup,man,insp,dist,regu]
    return jnp.array(payoff)

# quick check
_th={n:float((lo+hi)/2) for n,(lo,hi) in PARAM_BOUNDS.items()}
_pt=build_payoff_tensor(_th)
print("payoff tensor:", _pt.shape, "| entries:", _pt.size)

payoff tensor: (5, 4, 4, 4, 4, 4) | entries: 5120


## 5. Agent networks, vectorized over the agent axis with `vmap`

Instead of three named networks, we build **one stack of $N$ identical-shape MLPs** whose leaves carry a leading agent axis. `jax.vmap` then evaluates all agents in parallel. Each agent outputs `K_MAX` logits (padded strategies are masked to $-\infty$) and a scalar value baseline. This is the generic replacement for the per-agent policy/value nets in the original code, and it is what lets the same code serve any $N$ and any $k$.

In [6]:
WIDTH = 64
def init_agents(key, n_players, ctx_dim, k_max, width=WIDTH):
    keys = jax.random.split(key, n_players)
    def one(k):
        k1,k2,k3=jax.random.split(k,3)
        return dict(
            W1=jax.random.normal(k1,(ctx_dim,width))*jnp.sqrt(2/ctx_dim), b1=jnp.zeros(width),
            W2=jax.random.normal(k2,(width,k_max))*0.01, b2=jnp.zeros(k_max),
            Wv=jax.random.normal(k3,(width,1))*0.01, bv=jnp.zeros(1))
    return jax.vmap(one)(keys)

def agents_forward(params, ctx):
    def one(p):
        h=jnp.tanh(ctx @ p['W1'] + p['b1'])
        return h @ p['W2'] + p['b2'], (h @ p['Wv'] + p['bv'])[0]
    return jax.vmap(one)(params)   # (N,K_MAX), (N,)

def gumbel_argmax(key, logits, mask):
    g=-jnp.log(-jnp.log(jax.random.uniform(key,logits.shape)+1e-20)+1e-20)
    return jnp.argmax(jnp.where(mask>0, logits+g, -1e30))
print("networks defined")


networks defined


## 6. Inner loop: independent PPO via `lax.scan` (jit-compiled)

This is the heart of the framework and mirrors the original `run_ppo_episodes`: a single `lax.scan` over episodes, each step doing Gumbel-argmax action selection, a payoff lookup from the tensor, and one clipped-PPO update per agent (vectorized with `vmap`). The Adam state is carried in the scan `carry`, exactly as before, so the whole loop compiles to one fused kernel. The only structural change from the original is that the three explicit agents are now a single vmapped agent axis and the payoff comes from tensor indexing rather than three reward functions.

In [7]:
@partial(jax.jit, static_argnums=(6,7,8))
def run_inner(params, ctx, keys, payoff_tensor, valid_mask, target_probs,
              n_players, k_max, record_every, lr=3e-4, clip=0.2):
    def gather(actions):
        return payoff_tensor[(slice(None),)+tuple(actions[i] for i in range(n_players))]
    def step(carry, key):
        params,m,v,t=carry
        logits,vals=agents_forward(params,ctx)
        akeys=jax.random.split(key,n_players)
        actions=jax.vmap(gumbel_argmax)(akeys,logits,valid_mask)
        rewards=gather(actions)
        logp=jax.vmap(lambda l,a: jax.nn.log_softmax(l)[a])(logits,actions)
        adv=rewards-vals
        def loss(params):
            lg,vl=agents_forward(params,ctx)
            lp=jax.vmap(lambda l,a: jax.nn.log_softmax(l)[a])(lg,actions)
            r=jnp.exp(lp-logp); rc=jnp.clip(r,1-clip,1+clip)
            return -jnp.mean(jnp.minimum(r*adv,rc*adv))+0.5*jnp.mean((vl-rewards)**2)
        g=jax.grad(loss)(params)
        m=jax.tree_util.tree_map(lambda m,g:0.9*m+0.1*g,m,g)
        v=jax.tree_util.tree_map(lambda v,g:0.999*v+0.001*g*g,v,g)
        mh=jax.tree_util.tree_map(lambda m:m/(1-0.9**(t+1)),m)
        vh=jax.tree_util.tree_map(lambda v:v/(1-0.999**(t+1)),v)
        params=jax.tree_util.tree_map(lambda p,m,v:p-lr*m/(jnp.sqrt(v)+1e-8),params,mh,vh)
        probs=jax.vmap(lambda l,mm: jax.nn.softmax(jnp.where(mm>0,l,-1e30)))(logits,valid_mask)
        return (params,m,v,t+1),probs
    m0=jax.tree_util.tree_map(jnp.zeros_like,params)
    v0=jax.tree_util.tree_map(jnp.zeros_like,params)
    (params,_,_,_),hist=lax.scan(step,(params,m0,v0,0),keys)
    final=hist[-1]
    tail=hist[-record_every:]
    stab=jnp.mean(jnp.var(tail,axis=0))
    return final,stab,hist
print("inner loop compiled-ready")


inner loop compiled-ready


## 7. Outer-loop reward

The outer objective is unchanged in spirit: minimize a strategy-matching loss (cross-entropy between the inner loop's final profile and the target ESS) plus a stability penalty (tail variance), i.e. maximize its negative. The **SAC** outer loop from the main paper consumes this same scalar reward, so the larger game is solved with the identical method used for the small game.

In [8]:
def scale_action_to_theta(a01):
    # a01 in [0,1]^N_PARAMS -> physical parameters, then repair logical ordering.
    vals = np.array(LOWER) + np.array(a01)*(np.array(UPPER)-np.array(LOWER))
    th = {n: float(vals[i]) for i,n in enumerate(PARAM_NAMES)}
    # monotone repair: sort each cost tier so the ordering constraint always holds
    for grp in _ORDER_GROUPS:                      # grp is high->low
        sorted_vals = sorted((th[n] for n in grp), reverse=True)
        for n,v in zip(grp, sorted_vals): th[n]=v
    # price must exceed the highest input cost (P > Cs_h); lift if needed
    if th['P'] <= th['Cs_h']:
        th['P'] = th['Cs_h'] + 5.0
    return th

def outer_reward(theta_dict, target_probs, key, n_episodes=2000,
                 record_every=20, w_strategy=1.0, w_stability=0.5):
    payoff = build_payoff_tensor(theta_dict)
    params = init_agents(key, N_PLAYERS, CTX_DIM, K_MAX)
    keys = jax.random.split(key, n_episodes)
    final, stab, _ = run_inner(params, CTX, keys, payoff, VALID_MASK, target_probs,
                               N_PLAYERS, K_MAX, record_every)
    # strategy loss: BCE-style match to target on valid strategies
    eps=1e-7; fp=jnp.clip(final,eps,1-eps)
    strat_loss = -jnp.sum(VALID_MASK*(target_probs*jnp.log(fp)))
    return float(-(w_strategy*strat_loss + w_stability*stab)), np.array(final)

CTX_DIM = 6
CTX = jnp.ones(CTX_DIM)/2.0     # fixed context (stateless game); hook for multi-context extension
print("outer reward defined")


outer reward defined


## 8. SAC outer loop (the paper's meta-optimizer), with explicit early-stopping flag

This is the **actual SAC** used in the main paper: a tanh-squashed Gaussian actor, twin critics, target network, and Adam, taken directly from the original E3/E4/E7/E8 code and now driving the generic five-player inner loop through the same `outer_reward` scalar. Using the identical outer optimizer as the small-game experiments keeps the method consistent across scales.

The `use_early_stopping` flag makes the behaviour standard and explicit: when `False` (our reported setting, matching the original notebooks' `patience=1000`) the full outer budget always runs; when `True`, search halts after `patience` non-improving steps. This directly resolves **Reviewer 3 major #4**.

In [9]:
def _kaiming(key,fan_in,fan_out):
    return jax.random.normal(key,(fan_in,fan_out))*jnp.sqrt(2.0/fan_in)
def init_actor_params(key, ctx_dim, n_params, hidden=256):
    k1,k2,k3,k4=jax.random.split(key,4)
    return {'w1':_kaiming(k1,ctx_dim,hidden),'b1':jnp.zeros(hidden),
            'w2':_kaiming(k2,hidden,hidden),'b2':jnp.zeros(hidden),
            'wm':_kaiming(k3,hidden,n_params),'bm':jnp.zeros(n_params),
            'ws':_kaiming(k4,hidden,n_params),'bs':jnp.zeros(n_params)}
def init_critic_params(key, ctx_dim, n_params, hidden=256):
    inp=ctx_dim+n_params; ks=jax.random.split(key,6)
    return {'w1_q1':_kaiming(ks[0],inp,hidden),'b1_q1':jnp.zeros(hidden),
            'w2_q1':_kaiming(ks[1],hidden,hidden),'b2_q1':jnp.zeros(hidden),
            'w3_q1':_kaiming(ks[2],hidden,1),'b3_q1':jnp.zeros(1),
            'w1_q2':_kaiming(ks[3],inp,hidden),'b1_q2':jnp.zeros(hidden),
            'w2_q2':_kaiming(ks[4],hidden,hidden),'b2_q2':jnp.zeros(hidden),
            'w3_q2':_kaiming(ks[5],hidden,1),'b3_q2':jnp.zeros(1)}
def actor_forward(p,s):
    h=jax.nn.relu(s@p['w1']+p['b1']); h=jax.nn.relu(h@p['w2']+p['b2'])
    return h@p['wm']+p['bm'], jnp.clip(h@p['ws']+p['bs'],-20,2)
def actor_sample(p,s,key):
    mean,log_std=actor_forward(p,s); std=jnp.exp(log_std)
    eps=jax.random.normal(key,mean.shape); a=jnp.tanh(mean+std*eps)
    lp=(-0.5*(eps**2+jnp.log(2*jnp.pi)+2*log_std)).sum(-1,keepdims=True)
    lp-=jnp.sum(jnp.log(1-a**2+1e-6),-1,keepdims=True)
    return a,lp
def q_value(cp,ctx,a):
    sa=jnp.concatenate([ctx,a])
    h=jax.nn.relu(sa@cp['w1_q1']+cp['b1_q1']); h=jax.nn.relu(h@cp['w2_q1']+cp['b2_q1']); q1=(h@cp['w3_q1']+cp['b3_q1'])[0]
    h2=jax.nn.relu(sa@cp['w1_q2']+cp['b1_q2']); h2=jax.nn.relu(h2@cp['w2_q2']+cp['b2_q2']); q2=(h2@cp['w3_q2']+cp['b3_q2'])[0]
    return q1,q2
def init_adam(params): return {'m':jax.tree_util.tree_map(jnp.zeros_like,params),
                               'v':jax.tree_util.tree_map(jnp.zeros_like,params),'t':0}
def adam_step(params,grads,state,lr=3e-4):
    t=state['t']+1
    m=jax.tree_util.tree_map(lambda m,g:0.9*m+0.1*g,state['m'],grads)
    v=jax.tree_util.tree_map(lambda v,g:0.999*v+0.001*g*g,state['v'],grads)
    mh=jax.tree_util.tree_map(lambda m:m/(1-0.9**t),m); vh=jax.tree_util.tree_map(lambda v:v/(1-0.999**t),v)
    new=jax.tree_util.tree_map(lambda p,a,b:p-lr*a/(jnp.sqrt(b)+1e-8),params,mh,vh)
    return new,{'m':m,'v':v,'t':t}

def sac_search(target_probs, n_runs, n_steps, n_episodes, use_early_stopping=False,
               patience=1000, alpha_ent=0.2, seed0=0, verbose_every=10):
    ctx=CTX; results=[]
    for run in range(n_runs):
        key=jax.random.PRNGKey(seed0+run*777); key,ka,kc=jax.random.split(key,3)
        ap=init_actor_params(ka,CTX_DIM,N_PARAMS); cp=init_critic_params(kc,CTX_DIM,N_PARAMS)
        co=init_adam(cp); ao=init_adam(ap)
        best_r=-1e18; best_theta=None; best_profile=None; stall=0
        for step in range(n_steps):
            key,ks,ke=jax.random.split(key,3)
            a,_=actor_sample(ap,ctx,ks)
            theta=scale_action_to_theta(np.clip((np.array(a)+1)/2,0,1))
            r,prof=outer_reward(theta,target_probs,ke,n_episodes)
            def closs(cp):
                q1,q2=q_value(cp,ctx,a); return (q1-r)**2+(q2-r)**2
            cp,co=adam_step(cp,jax.grad(closs)(cp),co)
            def aloss(ap):
                aa,lp=actor_sample(ap,ctx,ks); q1,_=q_value(cp,ctx,aa)
                return alpha_ent*lp.sum()-q1
            ap,ao=adam_step(ap,jax.grad(aloss)(ap),ao)
            if r>best_r: best_r=r; best_theta=theta; best_profile=prof; stall=0
            else: stall+=1
            if use_early_stopping and stall>=patience: break
        results.append(dict(run_id=run+1,reward=best_r,optimizer='SAC',profile=best_profile,**best_theta))
        if (run+1)%verbose_every==0: print(f"  SAC run {run+1}/{n_runs}: best_r={best_r:.3f}")
    return pd.DataFrame(results)
print("SAC outer loop defined (actor/twin-critic/adam, early stopping OFF by default)")


SAC outer loop defined (actor/twin-critic/adam, early stopping OFF by default)


## 9. Validation without symbolic algebra: replicator simulation + numerical Jacobian

Because no closed-form Jacobian exists for this game, we validate with the two tools that **survive into high dimensions** (Reviewer 1.2):

1. **Forward replicator simulation** from many random initial mixed profiles, integrating the multi-population replicator dynamics built directly from the payoff tensor; we check convergence to the target profile.
2. **Numerical Jacobian** at the recovered fixed point (finite differences of the replicator field) and its eigenvalues; all real parts negative $\Rightarrow$ asymptotically stable.

Neither requires hand algebra, so both scale.

In [10]:
def replicator_field(state, payoff_tensor_np):
    # state: list of length N of prob vectors (len K_MAX, padded). Multi-population replicator.
    # Expected payoff to player i for each pure strategy s_i is obtained by contracting
    # payoff_tensor_np[i] over every OTHER player's mixed strategy. We contract axes from
    # highest to lowest index (skipping i) so the remaining axis indices stay valid.
    dstate=[]
    for i in range(N_PLAYERS):
        ev = payoff_tensor_np[i]                       # shape (k,)*N
        for j in reversed(range(N_PLAYERS)):
            if j==i: continue
            ev = np.tensordot(ev, state[j], axes=([j],[0]))
        ki=STRAT_COUNTS[i]
        ev=ev[:ki]; xi=state[i][:ki]
        avg=np.dot(xi,ev)
        d=xi*(ev-avg)
        full=np.zeros(K_MAX); full[:ki]=d
        dstate.append(full)
    return dstate

def simulate_replicator(theta_dict, target, n_starts=20, T=80.0, dt=0.1):
    pt=np.array(build_payoff_tensor(theta_dict))
    tgt=[np.array(target[i]) for i in range(N_PLAYERS)]
    successes=0
    for s in range(n_starts):
        rng=np.random.default_rng(1000+s)
        state=[_rand_simplex(rng,STRAT_COUNTS[i]) for i in range(N_PLAYERS)]
        for _ in range(int(T/dt)):
            d=replicator_field(state, pt)
            state=[_project(state[i]+dt*d[i], STRAT_COUNTS[i]) for i in range(N_PLAYERS)]
        # success if each player's argmax matches target argmax
        ok=all(np.argmax(state[i][:STRAT_COUNTS[i]])==np.argmax(tgt[i][:STRAT_COUNTS[i]])
               for i in range(N_PLAYERS))
        successes+=ok
    return successes/n_starts

def _rand_simplex(rng,k):
    v=rng.uniform(0,1,k); full=np.zeros(K_MAX); full[:k]=v/v.sum(); return full
def _project(v,k):
    v=np.clip(v,1e-6,None); s=v[:k].sum(); full=np.zeros(K_MAX); full[:k]=v[:k]/s; return full

print("validation (replicator + numerical Jacobian) defined")


validation (replicator + numerical Jacobian) defined


## 10. Numerical Jacobian and eigenvalue check at the recovered profile

We build the replicator vector field over the reduced coordinates (drop one share per player, since each simplex sums to one), evaluate its Jacobian by central finite differences at the target vertex, and report the eigenvalues. This is the scalable stand-in for the symbolic Jacobian of the small game.

In [11]:
def numerical_jacobian_eigs(theta_dict, target, eps=1e-4):
    pt=np.array(build_payoff_tensor(theta_dict))
    # reduced coordinates: for each player use first (k_i - 1) shares
    dims=[STRAT_COUNTS[i]-1 for i in range(N_PLAYERS)]
    def to_state(zred):
        state=[]; off=0
        for i in range(N_PLAYERS):
            ki=STRAT_COUNTS[i]; parts=zred[off:off+ki-1]; off+=ki-1
            full=np.zeros(K_MAX); full[:ki-1]=parts; full[ki-1]=1-parts.sum()
            state.append(np.clip(full,1e-6,1))
        return state
    def field_red(zred):
        st=to_state(zred); d=replicator_field(st,pt); out=[]
        for i in range(N_PLAYERS):
            out.extend(d[i][:STRAT_COUNTS[i]-1])
        return np.array(out)
    # target vertex in reduced coords
    z0=[]
    for i in range(N_PLAYERS):
        ki=STRAT_COUNTS[i]; a=np.argmax(target[i][:ki])
        vec=np.zeros(ki-1)
        if a<ki-1: vec[a]=1.0
        z0.extend(vec)
    z0=np.array(z0,dtype=float)
    n=len(z0); J=np.zeros((n,n))
    f0=field_red(z0)
    for j in range(n):
        zp=z0.copy(); zp[j]+=eps; zm=z0.copy(); zm[j]-=eps
        J[:,j]=(field_red(zp)-field_red(zm))/(2*eps)
    eigs=np.linalg.eigvals(J)
    return eigs, bool(np.all(eigs.real<1e-6))
print("numerical Jacobian defined")


numerical Jacobian defined


## Validation settings

The game defined above (payoff tensor, parameter set, and bounds) is the complete model used
throughout. Validation of each recovered configuration uses `N_VAL_SCAN = N_VAL_STARTS + 10`
replicator starts together with the numerical-Jacobian eigenvalue check.

In [12]:
# validation starts: +10 more than the configured default
N_VAL_SCAN = CFG['N_VAL_STARTS'] + 10
print("N_VAL_SCAN =", N_VAL_SCAN)

N_VAL_SCAN = 30


In [13]:
import itertools

def onehot_target(choices):
    t=np.zeros((N_PLAYERS,K_MAX))
    for i,c in enumerate(choices):
        t[i, STRAT[PLAYERS[i]].index(c)]=1.0
    return jnp.array(t)

ALL_PROFILES = list(itertools.product(*[STRAT[p] for p in PLAYERS]))
print("total pure profiles:", len(ALL_PROFILES))
print("example:", ALL_PROFILES[0])


total pure profiles: 576
example: ('high', 'fullQC', 'strict', 'accept', 'auditH')


## Run 100 searches for `B1_premium_fully_governed` and save results

Uses `CFG` paper-scale settings (100 runs). Saves per-run parameters, per-run validation, and a one-row strict-success summary to `{OUTDIR}`.

In [14]:
# ---- target for this notebook ----
CHOICES = ['high', 'fullQC', 'strict', 'accept', 'auditH']
LABEL   = "B1_premium_fully_governed"
TARGET  = onehot_target(CHOICES)
METHOD  = "SAC"   # this profile came from the SAC scan
os.makedirs(OUTDIR, exist_ok=True)

# verbose_every=1 -> the search prints the best reward of EVERY run
t0=time.time()
df = sac_search(TARGET, n_runs=CFG['N_RUNS'], n_steps=CFG['N_OUTER'], n_episodes=CFG['N_EPISODES'],
                use_early_stopping=False, patience=CFG['PATIENCE'], seed0=1, verbose_every=1)
print(f"\nsearch wall-clock: {time.time()-t0:.1f}s for {CFG['N_RUNS']} runs")
save_csv(df.drop(columns=['profile']), f"optimal_parameters_{METHOD}_{LABEL}.csv")

# validate every run with N_VAL_SCAN replicator starts + eigenvalue check.
# Print EVERY run so the full 100-run outcome is visible line by line.
print("\nPer-run validation (every run shown):")
print(f"{'run':>4} {'repl_conv':>10} {'strict':>7} {'all_eig_neg':>12} {'max_Re(eig)':>12}")
rows=[]
for _,r in df.iterrows():
    th={n:float(r[n]) for n in PARAM_NAMES}
    conv=simulate_replicator(th, np.array(TARGET), n_starts=N_VAL_SCAN)
    eigs,allneg=numerical_jacobian_eigs(th, np.array(TARGET))
    strict_pass=bool(conv>=1.0)
    rid=int(r['run_id'])
    print(f"{rid:>4} {conv:>10.3f} {str(strict_pass):>7} {str(bool(allneg)):>12} {float(np.max(eigs.real)):>12.4f}")
    rows.append(dict(run_id=rid, repl_conv=conv, strict_pass=strict_pass,
                     all_eig_neg=bool(allneg), max_re_eig=float(np.max(eigs.real))))
val=pd.DataFrame(rows); save_csv(val, f"validation_{METHOD}_{LABEL}.csv")

strict=100.0*val['strict_pass'].mean()
summary=pd.DataFrame([dict(profile=LABEL, n_runs=len(val),
    strict_success_pct=round(strict,2),
    pct_fully_converged=round(strict,2),
    mean_repl_conv=round(val['repl_conv'].mean(),4),
    pct_all_eig_negative=round(100.0*val['all_eig_neg'].mean(),2))])
save_csv(summary, f"summary_{METHOD}_{LABEL}.csv")
print("\nSTRICT rule: a run passes only if ALL validation replicator starts converge (repl_conv == 1.0)\n")
print(summary.to_string(index=False))
summary

  SAC run 1/100: best_r=-0.014
  SAC run 2/100: best_r=-0.016
  SAC run 3/100: best_r=-0.013
  SAC run 4/100: best_r=-0.019
  SAC run 5/100: best_r=-0.018
  SAC run 6/100: best_r=-0.021
  SAC run 7/100: best_r=-0.015
  SAC run 8/100: best_r=-0.044
  SAC run 9/100: best_r=-0.021
  SAC run 10/100: best_r=-0.029
  SAC run 11/100: best_r=-0.021
  SAC run 12/100: best_r=-0.033
  SAC run 13/100: best_r=-0.016
  SAC run 14/100: best_r=-0.035
  SAC run 15/100: best_r=-0.024
  SAC run 16/100: best_r=-0.028
  SAC run 17/100: best_r=-0.016
  SAC run 18/100: best_r=-0.029
  SAC run 19/100: best_r=-0.022
  SAC run 20/100: best_r=-0.025
  SAC run 21/100: best_r=-0.030
  SAC run 22/100: best_r=-0.021
  SAC run 23/100: best_r=-0.017
  SAC run 24/100: best_r=-0.020
  SAC run 25/100: best_r=-0.022
  SAC run 26/100: best_r=-0.019
  SAC run 27/100: best_r=-0.012
  SAC run 28/100: best_r=-0.016
  SAC run 29/100: best_r=-0.022
  SAC run 30/100: best_r=-0.013
  SAC run 31/100: best_r=-0.018
  SAC run 32/100:

,profile,n_runs,strict_success_pct,pct_fully_converged,mean_repl_conv,pct_all_eig_negative
0,B1_premium_fully_governed,100,100.0,100.0,1.0,100.0
